In [7]:
import os
from dotenv import load_dotenv
load_dotenv()

if os.environ["GROQ_API_KEY"]:
    print("API Key Loaded...")

API Key Loaded...


In [8]:
from langchain_groq import ChatGroq

llm = ChatGroq(model = "openai/gpt-oss-120b", temperature = 0)

### **RAG Implementation With Your Own PDF DATA**

#### Step 1 : Preaparing Document

In [3]:
from langchain_community.document_loaders import PyPDFLoader

path = "../Data/topics.pdf"

loader = PyPDFLoader(path)

docs = loader.load()

docs

C:\Users\Samarth\AppData\Local\Temp\ipykernel_8764\95694974.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-10-16T16:11:02+05:30', 'msip_label_e65487c9-99ed-4cbc-93a8-0e9b1796bde5_enabled': 'true', 'msip_label_e65487c9-99ed-4cbc-93a8-0e9b1796bde5_setdate': '2026-10-16T10:40:49Z', 'msip_label_e65487c9-99ed-4cbc-93a8-0e9b1796bde5_method': 'Standard', 'msip_label_e65487c9-99ed-4cbc-93a8-0e9b1796bde5_name': 'defa4170-0d19-0005-0004-bc88714345d2', 'msip_label_e65487c9-99ed-4cbc-93a8-0e9b1796bde5_siteid': '03cb5f0c-1f82-4993-9621-36330f6309ec', 'msip_label_e65487c9-99ed-4cbc-93a8-0e9b1796bde5_actionid': '326da824-dc65-4853-a3f6-0d96d7bbe968', 'msip_label_e65487c9-99ed-4cbc-93a8-0e9b1796bde5_contentbits': '0', 'msip_label_e65487c9-99ed-4cbc-93a8-0e9b1796bde5_tag': '10, 3, 0, 1', 'author': 'Anuradha Swain', 'moddate': '2026-10-16T16:11:02+05:30', 'source': '../Data/topics.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}, page_content='SCIL Professional Skill

#### Step 2: Spliting The Document Into Chunks

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 600,
    chunk_overlap = 100
)

chunk = splitter.split_documents(docs)

chunk

[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-10-16T16:11:02+05:30', 'msip_label_e65487c9-99ed-4cbc-93a8-0e9b1796bde5_enabled': 'true', 'msip_label_e65487c9-99ed-4cbc-93a8-0e9b1796bde5_setdate': '2026-10-16T10:40:49Z', 'msip_label_e65487c9-99ed-4cbc-93a8-0e9b1796bde5_method': 'Standard', 'msip_label_e65487c9-99ed-4cbc-93a8-0e9b1796bde5_name': 'defa4170-0d19-0005-0004-bc88714345d2', 'msip_label_e65487c9-99ed-4cbc-93a8-0e9b1796bde5_siteid': '03cb5f0c-1f82-4993-9621-36330f6309ec', 'msip_label_e65487c9-99ed-4cbc-93a8-0e9b1796bde5_actionid': '326da824-dc65-4853-a3f6-0d96d7bbe968', 'msip_label_e65487c9-99ed-4cbc-93a8-0e9b1796bde5_contentbits': '0', 'msip_label_e65487c9-99ed-4cbc-93a8-0e9b1796bde5_tag': '10, 3, 0, 1', 'author': 'Anuradha Swain', 'moddate': '2026-10-16T16:11:02+05:30', 'source': '../Data/topics.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}, page_content='SCIL Professional Skill

#### Step 3: Creating Embeddings For Chunks

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="intfloat/e5-small-v2"
)

c:\Users\Samarth\Desktop\Learning\RAG-Journey\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 879.00it/s]


#### Step 4: Storing In Local Vector Database

In [2]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents = chunk,
    embedding = embedding_model,
    persist_directory = "../Vector/"    
)

NameError: name 'chunk' is not defined

In [1]:
content = vectorstore.similarity_search("What Is The Topics Of Gropu Number 16", k = 3)
content

NameError: name 'vectorstore' is not defined

#### Step 5: Re-Use Vector Database

In [4]:
from langchain_community.vectorstores import Chroma
vectorstore_persist = Chroma(
    persist_directory= "../Vector",
    embedding_function= embedding_model,
)

C:\Users\Samarth\AppData\Local\Temp\ipykernel_6200\1322464300.py:2: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore_persist = Chroma(


In [10]:
context = vectorstore_persist.similarity_search("What Is The Topics Of Gropu Number 16", k = 3)

In [11]:
response = llm.invoke(f"What is the description related to group 16 time management Use The Following Context To Answer: {context} And Give Pretty Answer")
print(response.content)

**Group 16 – Time‑Management (Presentation Description)**  

- **Topic focus:** Time‑management (the only topic assigned to Group 16).  
- **Slide count:** Exactly **6 slides** – no more, no fewer.  
- **Content requirements:**  
  1. **Slide 1 – Introduction & problem statement** – define the time‑management challenge you will address.  
  2. **Slide 2 – Background & research findings** – present relevant facts, data, or recent studies that illustrate why the issue matters.  
  3. **Slide 3 – Real‑world case study / industry example** – show a concrete example of time‑management practices (e.g., a company’s scheduling system, a personal productivity method, or a project‑management tool).  
  4. **Slide 4 – Critical analysis** – dissect the challenges, root causes, and impact of poor or effective time‑management.  
  5. **Slide 5 – Practical solutions & recommendations** – outline actionable strategies, tools, or professional skills that can improve time‑management, linking them to the